In [0]:
%sql
select * from vendor_performance.silver.vendor_sales_summary limit 10

In [0]:
dfsilver = spark.sql("select * from vendor_performance.silver.vendor_sales_summary")

In [0]:
from pyspark.sql.functions import col, coalesce, lit

dfsilver = dfsilver.withColumn("GrossProfit", coalesce(col("TotalSalesDollars"), lit(0)) - coalesce(col("TotalPurchaseDollars"), lit(0)))
dfsilver = dfsilver.withColumn("GrossProfitMargin", (col("GrossProfit") / col("TotalSalesDollars"))*100)
# StockTurnover - 1 if vendor brought and sold same qty, if sold less then < 1, if sold more then > 1 means vendor sold previously holding inventory
dfsilver = dfsilver.withColumn("StockTurnover", (col("TotalSalesQuantity") / col("TotalPurchaseQuantity")))
dfsilver = dfsilver.withColumn("SalesToPurchaseRatio", (col("TotalSalesDollars") / col("TotalPurchaseDollars")))

In [0]:
dfsilver.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("vendor_performance.gold.vendor_sales_summary")

In [0]:
%sql
select * from vendor_performance.gold.vendor_sales_summary limit 10